# SQL and Database Foundations: How Relational Engines Actually Work

Maps to `design3.md` Phase 2.

This notebook is a guided walkthrough of SQL reasoning and database internals. It is not a syntax reference. The goal is to build the mental models you need to write correct queries, read execution plans, design schemas, and reason about concurrent access — without guessing.

Industry context: trading data, time-series workloads, PostgreSQL, TimescaleDB. All examples use energy trading and market data scenarios.

Since we cannot run SQL directly in a Jupyter notebook, SQL examples are presented as formatted strings in code cells or as markdown code blocks. You will run these against a real PostgreSQL instance during weekend builds.

Working rule:

- read the explanation first
- study the SQL example
- modify it mentally or on paper before moving on
- write one short note in your own words after each section

## Learning Goals

By the end of this notebook, you should be able to:

- explain the logical execution order of a SQL query and why it matters for writing correct queries
- use all join types correctly and recognize common join anti-patterns
- write window functions for ranking, gap detection, running totals, and VWAP calculation
- use CTEs for readable multi-step query composition
- read an EXPLAIN ANALYZE output and identify the dominant bottleneck
- explain index types, composite index ordering, and when indexes help or hurt
- explain MVCC, transaction isolation, and their practical implications for concurrent pipelines
- explain WAL, crash recovery, and the connection to replication
- design partitioned tables for time-series workloads
- design a time-series schema with defensible index and partitioning choices
- explain normalization vs denormalization tradeoffs
- design a star schema with fact tables, dimension tables, and SCD Type 2

Working rule:

- read the walkthrough
- study the SQL
- modify the examples mentally or on paper
- write one short explanation in your own words before moving on

## 1. SQL Execution Order

Most people learn SQL syntax in the order you write it: `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY`. But the database engine does not process the query in that order. Understanding the logical processing order is the single most important thing for writing correct SQL without trial and error.

### The Logical Processing Order

```
1. FROM        -- identify the source tables and perform joins
2. WHERE       -- filter rows (before grouping)
3. GROUP BY    -- collapse rows into groups
4. HAVING      -- filter groups (after grouping)
5. SELECT      -- evaluate expressions and aliases
6. DISTINCT    -- remove duplicate rows
7. ORDER BY    -- sort the result
8. LIMIT       -- truncate the result
```

### Why This Matters

**You cannot use a SELECT alias in WHERE.** The WHERE clause executes before SELECT, so aliases defined in SELECT do not exist yet when WHERE runs.

**You cannot filter on window functions in WHERE.** Window functions execute as part of the SELECT phase, which happens after WHERE. If you need to filter on a window function result, you must wrap the query in a CTE or subquery.

**You can use a SELECT alias in ORDER BY.** ORDER BY executes after SELECT, so aliases are available there.

**HAVING filters groups, not rows.** If you want to filter individual rows, use WHERE. If you want to filter after aggregation (e.g., "only groups with more than 100 trades"), use HAVING.

### Common Confusion Points

| What you want | Wrong approach | Why it fails | Correct approach |
|---|---|---|---|
| Filter on an alias | `WHERE avg_price > 100` | Alias not yet defined | Use HAVING or wrap in CTE |
| Filter on window function | `WHERE ROW_NUMBER() OVER (...) = 1` | Window functions run in SELECT | Wrap in CTE, filter outer query |
| Use aggregate in WHERE | `WHERE COUNT(*) > 5` | Aggregates need GROUP BY first | Use HAVING |
| Reference column not in GROUP BY | `SELECT symbol, ts, AVG(price)` | ts is not grouped or aggregated | Add ts to GROUP BY or use a window function |

In [ ]:
# SQL Execution Order — Working Example
#
# This query BREAKS if you don't understand execution order.
# The intent: find symbols where the average trade price exceeds 50,
# and alias the average as avg_price.

broken_query = """
-- BROKEN: tries to use a SELECT alias in WHERE
SELECT symbol, AVG(price) AS avg_price
FROM trades
WHERE avg_price > 50        -- ERROR: column "avg_price" does not exist
GROUP BY symbol;
"""

fixed_query = """
-- FIXED: use HAVING to filter after GROUP BY + aggregation
SELECT symbol, AVG(price) AS avg_price
FROM trades
GROUP BY symbol
HAVING AVG(price) > 50      -- HAVING runs after GROUP BY, so aggregates work here
ORDER BY avg_price DESC;     -- ORDER BY runs after SELECT, so aliases work here
"""

# Why this breaks:
# WHERE executes at step 2, before GROUP BY (step 3) and SELECT (step 5).
# The alias avg_price is created in SELECT, so it does not exist during WHERE.
# HAVING executes at step 4, after GROUP BY, so aggregates are available.

print("=== BROKEN (alias in WHERE) ===")
print(broken_query)
print("=== FIXED (aggregate in HAVING) ===")
print(fixed_query)

# Try next:
# 1. Write a query that tries to filter on a window function in WHERE, then fix it with a CTE.
# 2. Write a query that uses an alias in ORDER BY and confirm it works.

## 2. Joins Deep Dive

Joins combine rows from two or more tables based on a related column. The join type determines what happens when a row in one table has no matching row in the other.

### Join Types

**INNER JOIN** — returns only rows that have a match in both tables. This is the default and most common join. Use it when you only want records that exist on both sides.

**LEFT JOIN (LEFT OUTER JOIN)** — returns all rows from the left table, plus matching rows from the right table. Where there is no match, the right side columns are NULL. Use it when you want to preserve all records from the primary table regardless of whether the secondary table has data.

**RIGHT JOIN (RIGHT OUTER JOIN)** — the mirror of LEFT JOIN. Returns all rows from the right table. In practice, most people rewrite RIGHT JOINs as LEFT JOINs by swapping the table order, which is easier to read.

**FULL OUTER JOIN** — returns all rows from both tables. Where there is no match on either side, the missing columns are NULL. Use it for full reconciliation: finding records that exist in A but not B, in B but not A, or in both.

**CROSS JOIN** — returns the Cartesian product: every row from the left table paired with every row from the right table. No join condition. Use it for generating combinations (e.g., all symbols crossed with all date ranges). Dangerous if used accidentally because it multiplies row counts.

### When to Use Each

| Join Type | Use Case | Data Pipeline Example |
|---|---|---|
| INNER | Only matched records | Enrich trades with instrument metadata (skip unknown instruments) |
| LEFT | Preserve primary, add optional data | Enrich trades with metadata, keep trades even if metadata is missing |
| FULL OUTER | Full reconciliation | Compare two vendor feeds to find discrepancies on both sides |
| CROSS | Generate combinations | Create a matrix of all symbols x all trading days for gap detection |

### The Reconciliation Pattern: LEFT JOIN WHERE NULL

This is one of the most important patterns in data engineering. It finds records that exist in table A but are missing from table B.

```sql
SELECT a.trade_id, a.symbol, a.ts
FROM vendor_a_trades a
LEFT JOIN vendor_b_trades b ON a.trade_id = b.trade_id
WHERE b.trade_id IS NULL;
```

How it works: the LEFT JOIN keeps all rows from vendor_a. For rows with no match in vendor_b, the b columns are NULL. The WHERE clause then filters to only those unmatched rows.

### Self-Joins

A self-join joins a table to itself. Use it when you need to compare rows within the same table.

Common use cases:
- compare a trade to the previous trade for the same symbol
- find pairs of trades that happened within the same time window
- hierarchical relationships (employee to manager)

### Common Anti-Patterns

**Accidental cross join.** If you forget the ON clause or use comma-separated FROM with no WHERE condition, you get a Cartesian product. On a 1M-row table joined to a 1M-row table, that is 1 trillion rows.

**Join on wrong key causing row explosion.** If the join key is not unique on one side, each row on the unique side gets duplicated for every matching row on the non-unique side. A 1:many relationship is fine if intended; it is a bug if you expected 1:1.

**Joining on NULLable columns without handling NULLs.** NULL = NULL evaluates to NULL (not TRUE) in SQL, so rows with NULL join keys will never match in a standard equality join.

In [ ]:
join_examples = {
    "reconciliation_find_missing": """
    -- Find trades in vendor A that are missing from vendor B.
    -- This is the core data reconciliation pattern.
    SELECT
        a.trade_id,
        a.symbol,
        a.price,
        a.ts
    FROM vendor_a_trades a
    LEFT JOIN vendor_b_trades b
        ON a.trade_id = b.trade_id
    WHERE b.trade_id IS NULL
    ORDER BY a.ts;
    """,

    "full_reconciliation": """
    -- Full reconciliation: find all discrepancies between two vendor feeds.
    -- Records only in A, only in B, and in both but with different prices.
    SELECT
        COALESCE(a.trade_id, b.trade_id) AS trade_id,
        a.price AS vendor_a_price,
        b.price AS vendor_b_price,
        CASE
            WHEN a.trade_id IS NULL THEN 'ONLY_IN_B'
            WHEN b.trade_id IS NULL THEN 'ONLY_IN_A'
            WHEN a.price <> b.price THEN 'PRICE_MISMATCH'
            ELSE 'MATCH'
        END AS reconciliation_status
    FROM vendor_a_trades a
    FULL OUTER JOIN vendor_b_trades b
        ON a.trade_id = b.trade_id
    WHERE a.trade_id IS NULL
       OR b.trade_id IS NULL
       OR a.price <> b.price;
    """,

    "self_join_consecutive_trades": """
    -- Self-join: compare each trade to the immediately previous trade
    -- for the same symbol to detect price jumps.
    -- (In practice you would use LAG for this, but self-join
    -- illustrates the concept.)
    SELECT
        t1.symbol,
        t1.ts AS current_ts,
        t1.price AS current_price,
        t2.ts AS prev_ts,
        t2.price AS prev_price,
        t1.price - t2.price AS price_change
    FROM trades t1
    INNER JOIN trades t2
        ON t1.symbol = t2.symbol
        AND t1.ts = (
            SELECT MIN(t3.ts)
            FROM trades t3
            WHERE t3.symbol = t1.symbol
              AND t3.ts > t2.ts
        );
    """,

    "cross_join_gap_matrix": """
    -- Cross join: generate all (symbol, trading_day) combinations
    -- then LEFT JOIN to actual trades to find gaps.
    SELECT
        s.symbol,
        d.trading_day,
        CASE WHEN t.symbol IS NULL THEN 'MISSING' ELSE 'OK' END AS status
    FROM (SELECT DISTINCT symbol FROM instruments WHERE is_active) s
    CROSS JOIN (
        SELECT generate_series('2026-01-01'::date, '2026-01-31'::date, '1 day') AS trading_day
    ) d
    LEFT JOIN trades t
        ON t.symbol = s.symbol
        AND t.ts::date = d.trading_day
    WHERE t.symbol IS NULL
    ORDER BY s.symbol, d.trading_day;
    """,
}

for name, sql in join_examples.items():
    print(f"=== {name} ===")
    print(sql)

# Try next:
# 1. Write a LEFT JOIN that enriches trades with instrument metadata.
# 2. Write a query to detect row explosion: join trades to a non-unique lookup and COUNT(*).
# 3. Rewrite the self-join example using LAG (covered in the next section).

## 3. Window Functions

Window functions are one of the most powerful features in SQL. They compute a value across a set of rows related to the current row, without collapsing the result into a single row per group. This is the key difference from GROUP BY: window functions add a computed column to each row while preserving all original rows.

### Mental Model

Think of a window function as: "for each row, look at a defined window of related rows, compute something, and attach the result to the current row."

The syntax:

```sql
function_name(...) OVER (
    PARTITION BY column    -- defines the groups (optional)
    ORDER BY column        -- defines the order within each group
    ROWS BETWEEN ...       -- defines the frame boundaries (optional)
)
```

### PARTITION BY vs GROUP BY

| Feature | GROUP BY | PARTITION BY (window) |
|---|---|---|
| Collapses rows | Yes — one output row per group | No — all original rows preserved |
| Can use with non-aggregated columns | No — everything must be grouped or aggregated | Yes — other columns stay as-is |
| Use case | Summary statistics | Per-row computation with group context |

### Ranking Functions

**ROW_NUMBER()** — assigns a unique sequential integer to each row within the partition. No ties; if two rows have equal values, one arbitrarily gets the lower number.

**RANK()** — like ROW_NUMBER but handles ties. Two rows with the same value get the same rank, and the next rank skips. Example: 1, 2, 2, 4.

**DENSE_RANK()** — like RANK but does not skip. Example: 1, 2, 2, 3.

When to use which:
- ROW_NUMBER: top-N per group (you want exactly N rows, no ties)
- RANK: ranking with gaps (competition-style ranking)
- DENSE_RANK: ranking without gaps (you want to know the Nth distinct value)

### LAG and LEAD

**LAG(column, offset, default)** — access a value from a previous row in the partition order. Offset defaults to 1.

**LEAD(column, offset, default)** — access a value from a subsequent row.

These are essential for:
- computing row-over-row changes (price change, volume delta)
- gap detection (time between consecutive events)
- comparing current value to previous value

### Window Frames: ROWS BETWEEN

By default, with ORDER BY specified, the frame is `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` (running total from start to current row).

Common frame specifications:

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW   -- running total
ROWS BETWEEN 4 PRECEDING AND CURRENT ROW           -- 5-row moving average
ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING            -- 3-row centered average
ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING  -- entire partition
```

### Industry Use Cases

- **Top-N per group**: "give me the 3 most recent trades per symbol" using ROW_NUMBER
- **Gap detection**: "find trades where the gap to the previous trade exceeds 5 minutes" using LAG
- **Running totals**: cumulative volume throughout the trading day
- **Moving averages**: 5-minute or 20-period moving average for smoothing
- **VWAP (Volume-Weighted Average Price)**: cumulative sum of (price * volume) / cumulative sum of volume

In [ ]:
window_examples = {
    "top_n_per_symbol": """
    -- Top 3 most recent trades per symbol using ROW_NUMBER.
    -- ROW_NUMBER guarantees exactly 3 rows per symbol, no ties.
    -- Note: we MUST use a CTE because you cannot filter on
    -- a window function in WHERE (execution order: WHERE runs before SELECT).
    WITH ranked AS (
        SELECT
            trade_id,
            symbol,
            price,
            volume,
            ts,
            ROW_NUMBER() OVER (
                PARTITION BY symbol
                ORDER BY ts DESC
            ) AS rn
        FROM trades
    )
    SELECT trade_id, symbol, price, volume, ts
    FROM ranked
    WHERE rn <= 3
    ORDER BY symbol, ts DESC;
    """,

    "ranking_comparison": """
    -- Compare ROW_NUMBER, RANK, and DENSE_RANK on the same data.
    -- Suppose two trades for AAPL have the same price.
    SELECT
        symbol,
        price,
        ts,
        ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY price DESC) AS row_num,
        RANK()       OVER (PARTITION BY symbol ORDER BY price DESC) AS rank_val,
        DENSE_RANK() OVER (PARTITION BY symbol ORDER BY price DESC) AS dense_rank_val
    FROM trades
    WHERE symbol = 'AAPL'
    ORDER BY price DESC;

    -- If two rows have price = 105.50:
    -- ROW_NUMBER: 1, 2       (arbitrary tiebreak)
    -- RANK:       1, 1       (tie, then skip to 3)
    -- DENSE_RANK: 1, 1       (tie, next is 2)
    """,

    "gap_detection_with_lag": """
    -- Gap detection: find trades where the time since the previous trade
    -- for the same symbol exceeds 5 minutes.
    -- This is critical for monitoring data feed health.
    SELECT
        symbol,
        ts,
        LAG(ts) OVER (PARTITION BY symbol ORDER BY ts) AS prev_ts,
        ts - LAG(ts) OVER (PARTITION BY symbol ORDER BY ts) AS gap_interval
    FROM trades
    WHERE ts >= NOW() - INTERVAL '1 day'
      AND ts - LAG(ts) OVER (PARTITION BY symbol ORDER BY ts) > INTERVAL '5 minutes';

    -- NOTE: this query is actually BROKEN due to execution order.
    -- The window function in WHERE will fail. Correct version:
    WITH trade_gaps AS (
        SELECT
            symbol,
            ts,
            LAG(ts) OVER (PARTITION BY symbol ORDER BY ts) AS prev_ts
        FROM trades
        WHERE ts >= NOW() - INTERVAL '1 day'
    )
    SELECT
        symbol,
        ts,
        prev_ts,
        ts - prev_ts AS gap_interval
    FROM trade_gaps
    WHERE ts - prev_ts > INTERVAL '5 minutes'
    ORDER BY gap_interval DESC;
    """,

    "moving_average": """
    -- 5-trade moving average of price per symbol.
    -- Uses ROWS BETWEEN to define a sliding window of 5 rows.
    SELECT
        symbol,
        ts,
        price,
        AVG(price) OVER (
            PARTITION BY symbol
            ORDER BY ts
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ) AS ma_5,
        COUNT(*) OVER (
            PARTITION BY symbol
            ORDER BY ts
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ) AS window_size
    FROM trades
    ORDER BY symbol, ts;
    -- window_size tells you how many rows are in the window.
    -- For the first 4 rows of each partition, the window is smaller than 5.
    """,

    "vwap_calculation": """
    -- VWAP (Volume-Weighted Average Price) — cumulative within each day.
    -- VWAP = cumulative(price * volume) / cumulative(volume)
    -- This is one of the most common trading analytics calculations.
    SELECT
        symbol,
        ts,
        price,
        volume,
        SUM(price * volume) OVER (
            PARTITION BY symbol, ts::date
            ORDER BY ts
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_pv,
        SUM(volume) OVER (
            PARTITION BY symbol, ts::date
            ORDER BY ts
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_vol,
        SUM(price * volume) OVER (
            PARTITION BY symbol, ts::date
            ORDER BY ts
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) /
        NULLIF(SUM(volume) OVER (
            PARTITION BY symbol, ts::date
            ORDER BY ts
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 0) AS vwap
    FROM trades
    ORDER BY symbol, ts;
    """,
}

for name, sql in window_examples.items():
    print(f"=== {name} ===")
    print(sql)

# Try next:
# 1. Write a query using LEAD to find the next trade's price for each row.
# 2. Compute a running total of volume per symbol per day.
# 3. Use DENSE_RANK to find the top 3 distinct price levels per symbol.

## 4. CTEs (Common Table Expressions)

A CTE is a named temporary result set that exists for the duration of a single query. Think of it as defining a named subquery at the top of your statement, then referencing it below.

### Why CTEs Exist

**Readability.** Complex queries with nested subqueries become unreadable fast. CTEs let you break a multi-step analysis into named, sequential stages that read top-to-bottom.

**Window function filtering.** As we saw in the execution order section, you cannot filter on a window function in WHERE. The standard pattern is: compute the window function in a CTE, then filter in the outer query.

**Reuse within a single query.** If you need the same intermediate result multiple times, a CTE lets you define it once and reference it multiple times. (Note: PostgreSQL may or may not materialize CTEs depending on version and usage. In PostgreSQL 12+, CTEs that are referenced once are typically inlined by the optimizer.)

### Syntax

```sql
WITH step_one AS (
    SELECT ...
),
step_two AS (
    SELECT ...
    FROM step_one
    WHERE ...
)
SELECT ...
FROM step_two;
```

### Recursive CTEs

Recursive CTEs handle hierarchical or graph-like data: organizational trees, category hierarchies, or iterative computations.

The structure:

```sql
WITH RECURSIVE cte_name AS (
    -- Base case: the starting rows
    SELECT ...
    UNION ALL
    -- Recursive case: joins back to cte_name
    SELECT ...
    FROM cte_name
    JOIN other_table ON ...
    WHERE termination_condition
)
SELECT * FROM cte_name;
```

Use cases in data engineering:
- traversing category hierarchies (instrument classification trees)
- generating date series when generate_series is not available
- graph traversal for dependency chains

In [ ]:
cte_examples = {
    "multi_step_analysis": """
    -- Multi-step analysis: find symbols where the latest trade price
    -- is more than 2 standard deviations above the daily average.
    -- Step 1: compute daily stats per symbol.
    -- Step 2: get the latest trade per symbol.
    -- Step 3: compare and flag anomalies.
    WITH daily_stats AS (
        SELECT
            symbol,
            ts::date AS trading_day,
            AVG(price) AS avg_price,
            STDDEV(price) AS stddev_price,
            COUNT(*) AS trade_count
        FROM trades
        WHERE ts >= CURRENT_DATE
        GROUP BY symbol, ts::date
    ),
    latest_trade AS (
        SELECT DISTINCT ON (symbol)
            symbol,
            price AS latest_price,
            ts AS latest_ts
        FROM trades
        WHERE ts >= CURRENT_DATE
        ORDER BY symbol, ts DESC
    )
    SELECT
        lt.symbol,
        lt.latest_price,
        lt.latest_ts,
        ds.avg_price,
        ds.stddev_price,
        ds.trade_count,
        CASE
            WHEN ds.stddev_price > 0
                 AND lt.latest_price > ds.avg_price + 2 * ds.stddev_price
            THEN 'ANOMALY_HIGH'
            WHEN ds.stddev_price > 0
                 AND lt.latest_price < ds.avg_price - 2 * ds.stddev_price
            THEN 'ANOMALY_LOW'
            ELSE 'NORMAL'
        END AS status
    FROM latest_trade lt
    JOIN daily_stats ds ON lt.symbol = ds.symbol
    ORDER BY status, lt.symbol;
    """,

    "recursive_cte_date_series": """
    -- Recursive CTE: generate a date series.
    -- Useful when generate_series is not available or you need custom logic.
    WITH RECURSIVE date_series AS (
        -- Base case: start date
        SELECT DATE '2026-01-01' AS trading_day
        UNION ALL
        -- Recursive case: add one day, stop at end date
        SELECT trading_day + INTERVAL '1 day'
        FROM date_series
        WHERE trading_day < DATE '2026-01-31'
    )
    SELECT trading_day
    FROM date_series;
    """,

    "recursive_cte_hierarchy": """
    -- Recursive CTE: traverse an instrument classification hierarchy.
    -- Example: Energy > Power > Baseload > DE Baseload
    WITH RECURSIVE classification_tree AS (
        -- Base case: root categories (no parent)
        SELECT
            category_id,
            category_name,
            parent_id,
            1 AS depth,
            category_name::text AS path
        FROM instrument_categories
        WHERE parent_id IS NULL

        UNION ALL

        -- Recursive case: children
        SELECT
            c.category_id,
            c.category_name,
            c.parent_id,
            ct.depth + 1,
            ct.path || ' > ' || c.category_name
        FROM instrument_categories c
        JOIN classification_tree ct ON c.parent_id = ct.category_id
        WHERE ct.depth < 10  -- safety limit to prevent infinite recursion
    )
    SELECT category_id, category_name, depth, path
    FROM classification_tree
    ORDER BY path;
    """,
}

for name, sql in cte_examples.items():
    print(f"=== {name} ===")
    print(sql)

# Try next:
# 1. Rewrite the multi-step analysis without CTEs (nested subqueries) and compare readability.
# 2. Write a CTE that computes VWAP, then filters to only symbols where VWAP > 100.
# 3. Write a recursive CTE that generates trading hours (9:00 to 16:00 in 30-min increments).

## 5. EXPLAIN ANALYZE — Reading Query Plans

EXPLAIN shows you the execution plan the PostgreSQL planner has chosen. EXPLAIN ANALYZE actually runs the query and shows you the plan with real timing and row counts. This is how you move from guessing about performance to knowing.

### What EXPLAIN Shows

The output is a tree of plan nodes. Each node represents an operation. Data flows from leaf nodes (table scans) up through intermediate nodes (joins, sorts, aggregations) to the root node (final result).

### Key Scan Types

**Seq Scan (Sequential Scan)** — reads the entire table row by row. This is fine for small tables or when you need most of the rows. It is a problem on large tables when you only need a few rows — that means a useful index is missing.

**Index Scan** — uses an index to find the relevant rows, then fetches the actual row data from the heap (table storage). Good for selective queries that return a small percentage of the table.

**Index Only Scan** — satisfies the query entirely from the index without touching the heap. This is the fastest scan type. It requires a covering index (all columns the query needs are in the index) and that the visibility map is up to date (recent VACUUM).

**Bitmap Index Scan + Bitmap Heap Scan** — a two-step process. First, the index is scanned to build a bitmap of matching pages. Then the heap pages are fetched in physical order. PostgreSQL uses this when the selectivity is too low for a plain index scan but too high for a seq scan. Common with OR conditions or multiple index conditions.

### Join Strategies

**Nested Loop** — for each row in the outer table, scan the inner table for matches. Good when the outer side is small and the inner side has an index. Bad when both sides are large.

**Hash Join** — build a hash table from the smaller side, then probe it with rows from the larger side. Good for equi-joins on larger datasets. Requires memory for the hash table (work_mem).

**Merge Join** — both sides are sorted on the join key, then merged in a single pass. Good when both sides are already sorted (e.g., from an index) or when the dataset is large enough that sorting + single-pass merge beats hash join.

### Reading the Numbers

```
Sort  (cost=1234.56..1245.67 rows=1000 width=48) (actual time=12.3..13.5 rows=950 loops=1)
```

| Field | Meaning |
|---|---|
| `cost=1234.56..1245.67` | Estimated startup cost..total cost (arbitrary planner units) |
| `rows=1000` | Estimated number of rows the node will produce |
| `width=48` | Estimated average row width in bytes |
| `actual time=12.3..13.5` | Real wall-clock time in milliseconds (startup..total) |
| `rows=950` | Actual number of rows produced |
| `loops=1` | How many times this node was executed (important for nested loops) |

### How to Spot Problems

1. **Seq Scan on a large table with a selective WHERE** — missing index. The planner is reading millions of rows to find a few.

2. **Bad row estimates** — if estimated rows and actual rows differ by orders of magnitude, the planner may have chosen the wrong strategy. This often happens with correlated columns, stale statistics, or complex expressions. Fix: `ANALYZE` the table to update statistics.

3. **Sort spill to disk** — look for `Sort Method: external merge Disk: NNNNkB`. This means the sort did not fit in work_mem and spilled to disk. Fix: increase work_mem or reduce the data being sorted.

4. **Nested Loop with large outer side** — if the outer side has millions of rows and the inner side does a scan for each, the total work is enormous. The planner usually avoids this, but bad statistics can cause it.

5. **High loops count on inner nodes** — in a Nested Loop, the inner node executes once per outer row. Multiply actual time by loops to get the true cost.

In [ ]:
# EXPLAIN ANALYZE — Walking Through a Real Plan
#
# Below is a realistic EXPLAIN ANALYZE output for a query that finds
# recent trades for a specific symbol. We annotate each node.

explain_output = """
EXPLAIN ANALYZE
SELECT trade_id, symbol, price, volume, ts
FROM trades
WHERE symbol = 'AAPL'
  AND ts >= '2026-01-01'
  AND ts < '2026-02-01'
ORDER BY ts DESC
LIMIT 100;

-- PLAN OUTPUT (annotated):

Limit  (cost=0.56..234.12 rows=100 width=52) (actual time=0.045..0.312 rows=100 loops=1)
  ->  Index Scan Backward using idx_trades_symbol_ts on trades
        (cost=0.56..8901.23 rows=3812 width=52) (actual time=0.043..0.298 rows=100 loops=1)
        Index Cond: ((symbol = 'AAPL'::text) AND (ts >= '2026-01-01') AND (ts < '2026-02-01'))
Planning Time: 0.185 ms
Execution Time: 0.341 ms
"""

# Node-by-node reading:
plan_walkthrough = {
    "step_1_leaf_node": {
        "node": "Index Scan Backward using idx_trades_symbol_ts",
        "what_it_does": (
            "Uses the composite index (symbol, ts) to find rows matching "
            "symbol='AAPL' AND ts in [2026-01-01, 2026-02-01). "
            "Scans backward because the outer query wants ORDER BY ts DESC."
        ),
        "estimated_rows": 3812,
        "actual_rows": 100,
        "why_only_100": (
            "The parent LIMIT node stopped requesting rows after 100. "
            "The index scan estimated it COULD produce 3812 rows, "
            "but the LIMIT short-circuited after 100."
        ),
    },
    "step_2_root_node": {
        "node": "Limit",
        "what_it_does": "Stops after receiving 100 rows from the child node.",
        "actual_time_ms": 0.312,
    },
    "overall_assessment": {
        "is_this_plan_good": "Yes",
        "why": (
            "Index Scan (not Seq Scan) means we have a useful index. "
            "Backward scan avoids a separate Sort node. "
            "LIMIT short-circuits early. Total execution: 0.341ms."
        ),
    },
}

# Now contrast with a BAD plan for the same query without the index:
bad_explain_output = """
-- Same query, but imagine idx_trades_symbol_ts does not exist:

Sort  (cost=45678.90..45679.15 rows=3812 width=52) (actual time=892.1..892.3 rows=100 loops=1)
  Sort Key: ts DESC
  Sort Method: top-N heapsort  Memory: 35kB
  ->  Seq Scan on trades
        (cost=0.00..41234.00 rows=3812 width=52) (actual time=0.021..845.7 rows=3805 loops=1)
        Filter: ((symbol = 'AAPL') AND (ts >= '2026-01-01') AND (ts < '2026-02-01'))
        Rows Removed by Filter: 2996195
Planning Time: 0.102 ms
Execution Time: 892.5 ms
"""

bad_plan_analysis = {
    "problem": "Seq Scan on trades — reading 3M rows to find 3800",
    "rows_removed_by_filter": "2,996,195 rows read and discarded",
    "sort_overhead": "Had to sort 3800 rows to apply ORDER BY + LIMIT",
    "execution_time": "892ms vs 0.34ms with the index",
    "fix": "CREATE INDEX idx_trades_symbol_ts ON trades (symbol, ts);",
    "speedup": "~2600x faster with the index",
}

print("=== GOOD PLAN (with index) ===")
print(explain_output)
print("\n=== BAD PLAN (without index) ===")
print(bad_explain_output)
print("\n=== ANALYSIS ===")
for k, v in bad_plan_analysis.items():
    print(f"  {k}: {v}")

# Try next:
# 1. Write EXPLAIN ANALYZE for a JOIN query and predict which join strategy PostgreSQL would pick.
# 2. Identify what "Rows Removed by Filter" tells you about index quality.
# 3. Look for "Sort Method: external merge Disk" in your own query plans.

## 6. Indexes

An index is a separate data structure that the database maintains alongside the table data. It provides fast lookup paths so the engine can find rows without scanning the entire table. The tradeoff: indexes speed up reads but slow down writes because every INSERT, UPDATE, or DELETE must also update the index.

### B-tree Indexes (the default)

B-tree is the default index type in PostgreSQL. It stores keys in a sorted, balanced tree structure.

What B-tree is good for:
- equality lookups: `WHERE symbol = 'AAPL'`
- range queries: `WHERE ts >= '2026-01-01' AND ts < '2026-02-01'`
- sorted access: `ORDER BY ts` can be satisfied directly from the index
- prefix matching: `WHERE symbol LIKE 'AA%'` (but not `LIKE '%PL'`)

### Composite Indexes and Column Order

A composite index covers multiple columns. **Column order matters** because of the leftmost prefix rule.

Given: `CREATE INDEX idx_trades_symbol_ts ON trades (symbol, ts);`

| Query | Uses the index? | Why |
|---|---|---|
| `WHERE symbol = 'AAPL' AND ts >= '2026-01-01'` | Yes | Matches both columns left-to-right |
| `WHERE symbol = 'AAPL'` | Yes | Matches the leftmost prefix |
| `WHERE ts >= '2026-01-01'` | No (or poorly) | Skips the first column; cannot use the sorted structure efficiently |
| `ORDER BY symbol, ts` | Yes | Matches the index order |
| `ORDER BY ts, symbol` | No | Wrong order |

**Rule of thumb for composite index column order:**
1. Put equality columns first (columns used with `=`)
2. Put the range or sort column last (columns used with `>`, `<`, `BETWEEN`, `ORDER BY`)

Example: if you always query `WHERE symbol = ? ORDER BY ts DESC`, the index should be `(symbol, ts)`.

### Covering Indexes (INCLUDE)

A covering index contains all columns the query needs, so PostgreSQL never has to visit the heap table. This enables Index Only Scans.

```sql
-- The query needs: symbol, ts, price
-- The covering index:
CREATE INDEX idx_trades_covering ON trades (symbol, ts) INCLUDE (price);
```

Now a query like `SELECT price FROM trades WHERE symbol = 'AAPL' AND ts >= '...'` can be answered entirely from the index.

### Hash Indexes

Hash indexes support only equality lookups (`=`). They do not support range queries, sorting, or partial matching. In PostgreSQL 10+, hash indexes are WAL-logged and crash-safe. They can be slightly faster than B-tree for pure equality lookups on very large tables, but in practice B-tree is almost always the better default.

### When Indexes Hurt

- **Write-heavy workloads**: every INSERT/UPDATE/DELETE must update every index on the table. A table with 10 indexes on a high-throughput ingestion pipeline pays the cost 10 times per write.
- **Low selectivity columns**: indexing a boolean column (two distinct values) rarely helps because half the table matches each value. The planner will prefer a Seq Scan.
- **Wide indexes**: indexes on large text columns or many columns consume significant disk space and memory.
- **Too many indexes**: each index adds maintenance cost. Over-indexing is a real problem in production.

### Index Design Rules for Data Platform Work

1. Start with your most common query patterns, not with the schema
2. Create composite indexes that match your WHERE + ORDER BY patterns
3. Put equality columns before range columns
4. Consider INCLUDE for frequently selected columns to enable Index Only Scans
5. Monitor unused indexes with `pg_stat_user_indexes` and drop them
6. For time-series: `(symbol, ts)` is almost always the first index you need

In [ ]:
index_examples = {
    "create_composite_index": """
    -- Composite index for the most common query pattern:
    -- "give me recent trades for a symbol, ordered by time"
    CREATE INDEX idx_trades_symbol_ts ON trades (symbol, ts);

    -- This single index supports:
    -- WHERE symbol = 'AAPL' AND ts >= '...' ORDER BY ts
    -- WHERE symbol = 'AAPL' (leftmost prefix)
    -- ORDER BY symbol, ts (matches index order)
    """,

    "covering_index": """
    -- Covering index: include price and volume so the query
    -- never touches the heap table.
    CREATE INDEX idx_trades_covering
        ON trades (symbol, ts)
        INCLUDE (price, volume);

    -- Now this query uses Index Only Scan:
    SELECT price, volume
    FROM trades
    WHERE symbol = 'AAPL'
      AND ts >= '2026-01-01'
      AND ts < '2026-02-01';
    """,

    "partial_index": """
    -- Partial index: only index rows that match a condition.
    -- Useful when you frequently query a subset of the data.
    CREATE INDEX idx_trades_errors
        ON trades (ts)
        WHERE status = 'ERROR';

    -- This index is tiny compared to a full index on ts,
    -- and it makes queries like this very fast:
    SELECT * FROM trades WHERE status = 'ERROR' AND ts >= NOW() - INTERVAL '1 hour';
    """,

    "compare_with_without_index": """
    -- Compare query plans with and without the index:

    -- WITHOUT index:
    EXPLAIN ANALYZE SELECT * FROM trades WHERE symbol = 'AAPL' AND ts >= '2026-01-01';
    -- Expected: Seq Scan, Filter, high "Rows Removed by Filter"
    -- Execution time: hundreds of milliseconds on large tables

    -- Create the index:
    CREATE INDEX idx_trades_symbol_ts ON trades (symbol, ts);

    -- WITH index:
    EXPLAIN ANALYZE SELECT * FROM trades WHERE symbol = 'AAPL' AND ts >= '2026-01-01';
    -- Expected: Index Scan using idx_trades_symbol_ts
    -- Execution time: sub-millisecond
    """,

    "monitor_unused_indexes": """
    -- Find indexes that are never used (candidates for removal).
    -- Reduces write overhead on ingestion-heavy tables.
    SELECT
        schemaname,
        relname AS table_name,
        indexrelname AS index_name,
        idx_scan AS times_used,
        pg_size_pretty(pg_relation_size(indexrelid)) AS index_size
    FROM pg_stat_user_indexes
    WHERE idx_scan = 0
    ORDER BY pg_relation_size(indexrelid) DESC;
    """,
}

for name, sql in index_examples.items():
    print(f"=== {name} ===")
    print(sql)

# Try next:
# 1. Design a composite index for: WHERE venue = ? AND ts BETWEEN ? AND ? ORDER BY price.
# 2. Explain why (venue, ts, price) is better than (ts, venue, price) for that query.
# 3. Write a partial index for trades where volume > 10000 (large block trades).

## 7. MVCC (Multi-Version Concurrency Control)

MVCC is how PostgreSQL handles concurrent reads and writes without locking readers out. Understanding MVCC is essential for reasoning about what your queries see when multiple pipelines and analytics users hit the same tables simultaneously.

### The Core Idea

When a row is updated in PostgreSQL, the old version is not overwritten. Instead, a new version of the row is created. Each transaction sees a consistent snapshot of the database as of the time the transaction (or statement) started. Old versions are kept around until no active transaction needs them anymore.

This means:
- **Readers do not block writers.** A long-running analytics query does not prevent the ingestion pipeline from inserting new rows.
- **Writers do not block readers.** The ingestion pipeline inserting rows does not make the analytics query wait.
- **Writers block writers** only when two transactions try to modify the same row.

### Transaction Isolation Levels

PostgreSQL supports four isolation levels, but two are the most important in practice:

**READ COMMITTED (default)** — each statement within the transaction sees a snapshot as of the statement's start time. If another transaction commits between two statements in your transaction, the second statement will see the newly committed data. This is fine for most workloads.

**REPEATABLE READ** — the entire transaction sees a snapshot as of the transaction's start time. Even if other transactions commit during your transaction, you will not see their changes. Use this when you need consistent reads across multiple statements (e.g., generating a report that must be self-consistent).

**SERIALIZABLE** — the strictest level. PostgreSQL ensures that the result is equivalent to running transactions one at a time. If it detects a conflict, it will abort one transaction with a serialization failure. Use this when correctness requires it and you can handle retries.

**READ UNCOMMITTED** — PostgreSQL treats this as READ COMMITTED. There is no "dirty read" behavior in PostgreSQL.

### Practical Implications

**Scenario: analytics query during heavy ingestion.**

With READ COMMITTED (default): your analytics query sees all data committed before each statement starts. If you run two SELECTs in the same transaction, they might see different data if the ingestion pipeline committed rows between them.

With REPEATABLE READ: your entire analytics transaction sees a frozen snapshot. Both SELECTs see identical data, even if the pipeline committed rows in between. This is what you want for consistent reporting.

### Dead Tuples and VACUUM

MVCC creates a problem: old row versions (dead tuples) accumulate. They are invisible to new transactions but still occupy disk space.

**VACUUM** reclaims space from dead tuples. It marks the space as reusable for future inserts but does not shrink the table file on disk.

**VACUUM FULL** rewrites the entire table to reclaim space and shrink the file. It requires an exclusive lock on the table — do not run this on production during business hours.

**autovacuum** runs VACUUM automatically in the background. For most tables, the default settings are fine. For high-write tables (like a trades table receiving thousands of inserts per second), you may need to tune autovacuum to run more aggressively:

```sql
ALTER TABLE trades SET (
    autovacuum_vacuum_scale_factor = 0.01,    -- trigger at 1% dead tuples (default 20%)
    autovacuum_analyze_scale_factor = 0.005   -- update statistics more often
);
```

### Industry Relevance

In a trading data platform:
- The ingestion pipeline writes continuously (producer/consumer)
- Analytics users run long queries against the same tables
- MVCC ensures they do not block each other
- autovacuum must keep up with the write rate or the table bloats
- Monitoring dead tuple ratio (`pg_stat_user_tables.n_dead_tup`) is a key operational metric

In [ ]:
mvcc_examples = {
    "check_dead_tuples": """
    -- Monitor dead tuples and autovacuum status for your tables.
    -- High n_dead_tup relative to n_live_tup means autovacuum is falling behind.
    SELECT
        schemaname,
        relname AS table_name,
        n_live_tup,
        n_dead_tup,
        ROUND(100.0 * n_dead_tup / NULLIF(n_live_tup + n_dead_tup, 0), 2) AS dead_pct,
        last_vacuum,
        last_autovacuum,
        last_analyze,
        last_autoanalyze
    FROM pg_stat_user_tables
    WHERE relname IN ('trades', 'quotes', 'aggregates')
    ORDER BY n_dead_tup DESC;
    """,

    "isolation_level_demo": """
    -- Demonstrate REPEATABLE READ for consistent analytics.
    -- Session 1 (analytics):
    BEGIN TRANSACTION ISOLATION LEVEL REPEATABLE READ;
    SELECT COUNT(*) FROM trades WHERE ts::date = CURRENT_DATE;  -- sees 10000 rows
    -- ... time passes, ingestion pipeline inserts 500 more rows ...
    SELECT COUNT(*) FROM trades WHERE ts::date = CURRENT_DATE;  -- STILL sees 10000 rows
    COMMIT;

    -- With READ COMMITTED (default), the second SELECT would see 10500 rows.
    -- With REPEATABLE READ, both SELECTs see the same snapshot.
    """,

    "tune_autovacuum_for_high_write": """
    -- Tune autovacuum for a high-write trades table.
    -- Default: vacuum triggers at 20% dead tuples.
    -- For a table with 10M rows, that means 2M dead tuples before cleanup.
    -- For high-throughput ingestion, trigger much sooner:
    ALTER TABLE trades SET (
        autovacuum_vacuum_scale_factor = 0.01,     -- 1% dead tuples triggers vacuum
        autovacuum_vacuum_cost_delay = 2,           -- less delay between vacuum I/O ops
        autovacuum_analyze_scale_factor = 0.005     -- update stats more frequently
    );
    """,

    "check_long_running_transactions": """
    -- Long-running transactions prevent VACUUM from cleaning up dead tuples.
    -- This query finds transactions that have been open for more than 5 minutes.
    SELECT
        pid,
        now() - xact_start AS transaction_duration,
        state,
        query
    FROM pg_stat_activity
    WHERE xact_start IS NOT NULL
      AND now() - xact_start > INTERVAL '5 minutes'
    ORDER BY xact_start;
    """,
}

for name, sql in mvcc_examples.items():
    print(f"=== {name} ===")
    print(sql)

# Try next:
# 1. Check dead tuple ratios on your own PostgreSQL instance.
# 2. Open two psql sessions and demonstrate READ COMMITTED vs REPEATABLE READ.
# 3. Explain why a forgotten open transaction can cause table bloat.

## 8. WAL (Write-Ahead Logging)

WAL is the mechanism that makes PostgreSQL crash-safe. Understanding it at a practical level is important for anyone operating databases or designing data pipelines that depend on durability guarantees.

### What WAL Is

Before PostgreSQL modifies actual data files (the heap, indexes), it first writes a description of the change to the Write-Ahead Log. The WAL is a sequential, append-only log of all changes.

The rule is simple: **the WAL record must be flushed to disk before the corresponding data page change is considered durable.** This is the "write-ahead" part.

### Why WAL Exists: Crash Recovery

If PostgreSQL crashes (power failure, OOM kill, kernel panic), the data files on disk may be in an inconsistent state — some pages written, others not. On restart, PostgreSQL replays the WAL from the last checkpoint forward, reapplying all committed changes and discarding uncommitted changes. This brings the database back to a consistent state.

Without WAL, every data page modification would need to be flushed to disk immediately (random I/O, extremely slow). WAL converts random writes into sequential writes (fast), and the actual data pages are flushed lazily in the background.

### Key WAL Concepts

**Checkpoint** — a point in time where PostgreSQL guarantees all data pages are flushed to disk. After a checkpoint, the WAL segments before it are no longer needed for crash recovery. Checkpoints happen periodically (controlled by `checkpoint_timeout` and `max_wal_size`). More frequent checkpoints = faster crash recovery but more I/O overhead.

**fsync** — the system call that forces data to be written to physical storage. PostgreSQL uses fsync to ensure WAL records are durable. Setting `fsync = off` is faster but means you can lose committed data on crash. Never do this in production.

**WAL segments** — WAL is split into fixed-size segment files (default 16MB each). Old segments are recycled or removed after checkpoints.

### Connection to Replication

Streaming replication in PostgreSQL works by sending WAL records from the primary to replicas. The replica replays the same WAL records, maintaining an identical copy of the database.

This means:
- Replication is at the physical level (WAL bytes), not logical SQL statements
- The replica is always slightly behind the primary (replication lag)
- WAL archiving enables Point-In-Time Recovery (PITR): you can restore to any moment by replaying WAL up to that point

### Practical Meaning for Data Platform Work

- **Durability guarantee**: once PostgreSQL returns "COMMIT OK," the change is in the WAL on disk. Even if the server crashes immediately after, the data survives.
- **Checkpoint tuning**: on write-heavy ingestion workloads, you may need to increase `max_wal_size` to reduce checkpoint frequency and spread the I/O.
- **WAL volume**: high-throughput INSERT workloads generate significant WAL. Monitor WAL generation rate and ensure disk has enough space.
- **Replica lag**: if your analytics queries run on a read replica, they may see data that is seconds behind the primary. This is usually acceptable for analytics but not for real-time decisions.

In [ ]:
wal_examples = {
    "check_wal_settings": """
    -- Check current WAL and checkpoint settings.
    SELECT name, setting, unit, short_desc
    FROM pg_settings
    WHERE name IN (
        'wal_level',
        'max_wal_size',
        'min_wal_size',
        'checkpoint_timeout',
        'checkpoint_completion_target',
        'fsync',
        'synchronous_commit'
    )
    ORDER BY name;
    """,

    "check_replication_lag": """
    -- Check replication lag on the primary.
    -- Shows how far behind each replica is.
    SELECT
        client_addr,
        state,
        sent_lsn,
        write_lsn,
        flush_lsn,
        replay_lsn,
        pg_wal_lsn_diff(sent_lsn, replay_lsn) AS replay_lag_bytes,
        pg_size_pretty(pg_wal_lsn_diff(sent_lsn, replay_lsn)) AS replay_lag_pretty
    FROM pg_stat_replication;
    """,

    "check_wal_generation_rate": """
    -- Estimate WAL generation rate by comparing pg_current_wal_lsn over time.
    -- Run this twice, 60 seconds apart, and compute the difference.
    SELECT pg_current_wal_lsn() AS current_lsn,
           pg_walfile_name(pg_current_wal_lsn()) AS current_wal_file;
    """,
}

for name, sql in wal_examples.items():
    print(f"=== {name} ===")
    print(sql)

# Try next:
# 1. Check your PostgreSQL instance's WAL settings.
# 2. Explain the relationship between checkpoint_timeout and crash recovery time.
# 3. Explain why synchronous_commit = off is sometimes acceptable for ingestion workloads.

## 9. Partitioning

Partitioning splits a large table into smaller physical pieces (partitions) while presenting them as a single logical table. The main benefit is partition pruning: when your query includes a condition on the partition key, PostgreSQL can skip partitions entirely, reading only the relevant ones.

### Range Partitioning (by date)

This is by far the most common partitioning strategy for time-series data. Each partition holds data for a specific time range (day, week, month).

```sql
CREATE TABLE trades (
    trade_id    UUID NOT NULL,
    symbol      TEXT NOT NULL,
    price       NUMERIC(18,8) NOT NULL,
    volume      NUMERIC(18,8) NOT NULL,
    ts          TIMESTAMPTZ NOT NULL
) PARTITION BY RANGE (ts);

CREATE TABLE trades_2026_01 PARTITION OF trades
    FOR VALUES FROM ('2026-01-01') TO ('2026-02-01');
CREATE TABLE trades_2026_02 PARTITION OF trades
    FOR VALUES FROM ('2026-02-01') TO ('2026-03-01');
```

When you query `WHERE ts >= '2026-01-15' AND ts < '2026-02-01'`, PostgreSQL only scans `trades_2026_01`. The other partitions are pruned entirely.

### List Partitioning (by category)

Partition by discrete values like symbol, region, or venue.

```sql
CREATE TABLE trades_by_venue (
    trade_id UUID, symbol TEXT, price NUMERIC, venue TEXT, ts TIMESTAMPTZ
) PARTITION BY LIST (venue);

CREATE TABLE trades_ecn PARTITION OF trades_by_venue FOR VALUES IN ('ECN');
CREATE TABLE trades_otc PARTITION OF trades_by_venue FOR VALUES IN ('OTC');
```

### When Partitioning Helps

- **Large time-series tables** where queries always include a time range: partition pruning eliminates most of the data.
- **Data lifecycle management**: drop old partitions instead of DELETE (instant, no dead tuples, no VACUUM needed).
- **Parallel query**: PostgreSQL can scan multiple partitions in parallel.
- **Maintenance isolation**: VACUUM, REINDEX, and backups can operate on individual partitions.

### When Partitioning Hurts

- **Cross-partition queries** that do not include the partition key: PostgreSQL must scan all partitions, which can be slower than a single table with a good index.
- **Too many partitions**: PostgreSQL's planner has overhead proportional to the number of partitions. Hundreds of partitions are fine; tens of thousands can slow down planning.
- **Non-partition-key lookups**: if you partition by date but frequently query by symbol without a date filter, partitioning does not help and may hurt.
- **Unique constraints**: unique indexes must include the partition key. You cannot have a globally unique `trade_id` constraint across partitions unless `trade_id` includes the partition key or you use a different enforcement mechanism.

### Practical Decision Framework

| Question | Answer |
|---|---|
| Is the table large (tens of millions+ rows)? | If no, partitioning adds complexity without benefit |
| Do queries almost always include the partition key? | If no, partitioning will not help much |
| Do you need to drop old data regularly? | Partitioning makes this instant |
| Is the write rate very high? | Partitioning can distribute autovacuum work |

In [ ]:
partitioning_examples = {
    "create_partitioned_table": """
    -- Create a partitioned trades table by month.
    CREATE TABLE trades (
        trade_id    UUID        NOT NULL DEFAULT gen_random_uuid(),
        symbol      TEXT        NOT NULL,
        price       NUMERIC(18,8) NOT NULL,
        volume      NUMERIC(18,8) NOT NULL,
        side        TEXT        NOT NULL CHECK (side IN ('BUY', 'SELL')),
        venue       TEXT        NOT NULL,
        ts          TIMESTAMPTZ NOT NULL
    ) PARTITION BY RANGE (ts);

    -- Create monthly partitions.
    -- In production, automate partition creation ahead of time.
    CREATE TABLE trades_2026_01 PARTITION OF trades
        FOR VALUES FROM ('2026-01-01') TO ('2026-02-01');
    CREATE TABLE trades_2026_02 PARTITION OF trades
        FOR VALUES FROM ('2026-02-01') TO ('2026-03-01');
    CREATE TABLE trades_2026_03 PARTITION OF trades
        FOR VALUES FROM ('2026-03-01') TO ('2026-04-01');

    -- Create indexes on each partition (they are independent tables).
    -- PostgreSQL can auto-create indexes on partitions if you create
    -- the index on the parent table:
    CREATE INDEX idx_trades_symbol_ts ON trades (symbol, ts);
    -- This creates a matching index on every existing and future partition.
    """,

    "partition_pruning_demo": """
    -- This query only touches trades_2026_01 due to partition pruning.
    EXPLAIN ANALYZE
    SELECT symbol, price, ts
    FROM trades
    WHERE ts >= '2026-01-15' AND ts < '2026-02-01'
      AND symbol = 'AAPL';

    -- Expected plan shows:
    -- -> Index Scan on trades_2026_01 trades
    --    Index Cond: ...
    -- The other partitions (trades_2026_02, trades_2026_03) are not mentioned.
    """,

    "drop_old_partition": """
    -- Data lifecycle: drop an entire month of data instantly.
    -- This is much faster than DELETE + VACUUM on millions of rows.
    -- No dead tuples, no bloat, no VACUUM needed.
    DROP TABLE trades_2025_01;

    -- Or detach it first if you want to archive:
    ALTER TABLE trades DETACH PARTITION trades_2025_01;
    -- Now trades_2025_01 is a standalone table you can archive or drop.
    """,

    "automate_partition_creation": """
    -- Create partitions automatically using a helper function.
    -- Run this monthly via cron or pg_cron.
    CREATE OR REPLACE FUNCTION create_monthly_partition(
        parent_table TEXT,
        partition_date DATE
    ) RETURNS VOID AS $$
    DECLARE
        partition_name TEXT;
        start_date DATE;
        end_date DATE;
    BEGIN
        start_date := date_trunc('month', partition_date)::date;
        end_date := (start_date + INTERVAL '1 month')::date;
        partition_name := parent_table || '_' || to_char(start_date, 'YYYY_MM');

        EXECUTE format(
            'CREATE TABLE IF NOT EXISTS %I PARTITION OF %I
             FOR VALUES FROM (%L) TO (%L)',
            partition_name, parent_table, start_date, end_date
        );
    END;
    $$ LANGUAGE plpgsql;

    -- Usage: create partition for next month
    SELECT create_monthly_partition('trades', '2026-04-01'::date);
    """,
}

for name, sql in partitioning_examples.items():
    print(f"=== {name} ===")
    print(sql)

# Try next:
# 1. Design a partitioning scheme for a quotes table (billions of rows, queried by symbol + time).
# 2. Explain why you might partition by week instead of month for very high-volume data.
# 3. Write a query that would NOT benefit from partitioning and explain why.

## 10. Time-Series Schema Design

Time-series data has specific access patterns that drive schema design. The key insight is that most queries filter by entity (symbol) and time range, and data is almost always appended chronologically and rarely updated.

### Core Tables for Trading Data

**Trades** — individual executed transactions. Each row is one trade event with a price, volume, side, and timestamp. This is the highest-volume table.

**Quotes** — bid/ask snapshots. Higher frequency than trades (every tick update). Used for spread analysis and market depth.

**Bars (OHLCV)** — pre-aggregated candles at fixed intervals (1m, 5m, 1h, 1d). Lower volume but heavily queried for charting and analytics.

### Access Patterns That Drive Design

1. **Recent data for a symbol**: `WHERE symbol = ? AND ts >= NOW() - INTERVAL '1 hour'` — the most common pattern
2. **Historical range for a symbol**: `WHERE symbol = ? AND ts BETWEEN ? AND ?` — backtest and research queries
3. **Cross-symbol at a point in time**: `WHERE ts = ?` — less common, snapshot queries
4. **Aggregation over time**: `GROUP BY symbol, date_trunc('hour', ts)` — analytics rollups

Patterns 1 and 2 dominate. This means the primary index should be `(symbol, ts)`.

### Choosing Primary Keys

For time-series tables, the natural primary key is often `(symbol, ts)` or a generated UUID. Consider:

- **UUID primary key**: globally unique, good for distributed systems, but random UUIDs cause index fragmentation (random insertion points in the B-tree). Use UUIDv7 (time-ordered) if available, or a BIGSERIAL.
- **Composite primary key `(symbol, ts)`**: natural and efficient for the main access pattern, but requires that no two trades for the same symbol have the exact same timestamp (which can happen in practice). Add a sequence tiebreaker: `(symbol, ts, seq)`.

### TimescaleDB: Hypertables

TimescaleDB extends PostgreSQL with automatic time-based partitioning (hypertables). Instead of manually creating monthly partitions, you create a hypertable and TimescaleDB manages the chunks.

```sql
-- Convert a regular table into a hypertable.
-- Chunks are created automatically based on the time column.
SELECT create_hypertable('trades', 'ts', chunk_time_interval => INTERVAL '1 day');
```

Benefits over manual partitioning:
- automatic chunk creation (no need to create partitions ahead of time)
- built-in compression (columnar compression on old chunks)
- continuous aggregates (materialized views that update incrementally)
- retention policies (automatic old chunk deletion)

### Compression and Retention

```sql
-- Enable compression on chunks older than 7 days.
ALTER TABLE trades SET (
    timescaledb.compress,
    timescaledb.compress_segmentby = 'symbol',
    timescaledb.compress_orderby = 'ts'
);
SELECT add_compression_policy('trades', INTERVAL '7 days');

-- Drop chunks older than 90 days.
SELECT add_retention_policy('trades', INTERVAL '90 days');
```

The `compress_segmentby` and `compress_orderby` settings are critical. They determine how data is grouped and sorted within compressed chunks. For trading data, segmenting by symbol and ordering by time matches the primary query pattern.

In [ ]:
timeseries_schema = {
    "trades_table": """
    -- Trades table: the core time-series table for a trading platform.
    -- Every design choice is explained.
    CREATE TABLE trades (
        trade_id    UUID            NOT NULL DEFAULT gen_random_uuid(),
            -- UUID for global uniqueness across distributed systems.
            -- Consider UUIDv7 for time-ordered inserts (better index locality).
        symbol      TEXT            NOT NULL,
            -- Text, not FK to instruments, for ingestion speed.
            -- Enrich with instrument metadata via JOIN at query time.
        price       NUMERIC(18,8)   NOT NULL,
            -- NUMERIC for exact decimal arithmetic. Never use FLOAT for prices.
            -- 18 total digits, 8 after decimal: handles all currency precisions.
        volume      NUMERIC(18,8)   NOT NULL,
        side        TEXT            NOT NULL CHECK (side IN ('BUY', 'SELL')),
        venue       TEXT            NOT NULL,
        ts          TIMESTAMPTZ     NOT NULL
            -- TIMESTAMPTZ: always store with timezone. Internally stored as UTC.
            -- Partitioning key for time-based partitions.
    ) PARTITION BY RANGE (ts);

    -- Primary access pattern: recent trades for a symbol.
    CREATE INDEX idx_trades_symbol_ts ON trades (symbol, ts);
        -- Composite: symbol (equality) first, ts (range) second.
        -- Supports: WHERE symbol = ? AND ts >= ? ORDER BY ts

    -- Secondary: lookup by trade_id across all partitions.
    CREATE INDEX idx_trades_trade_id ON trades (trade_id);
        -- Needed for deduplication and individual trade lookups.

    -- Covering index for common analytics query.
    CREATE INDEX idx_trades_analytics ON trades (symbol, ts)
        INCLUDE (price, volume);
        -- Enables Index Only Scan for VWAP and aggregation queries.
    """,

    "quotes_table": """
    -- Quotes table: bid/ask snapshots, higher frequency than trades.
    CREATE TABLE quotes (
        symbol      TEXT            NOT NULL,
        bid         NUMERIC(18,8)   NOT NULL,
        ask         NUMERIC(18,8)   NOT NULL,
        bid_size    NUMERIC(18,8),
        ask_size    NUMERIC(18,8),
        venue       TEXT            NOT NULL,
        ts          TIMESTAMPTZ     NOT NULL
    ) PARTITION BY RANGE (ts);

    CREATE INDEX idx_quotes_symbol_ts ON quotes (symbol, ts);
    """,

    "bars_table": """
    -- Bars (OHLCV): pre-aggregated candles.
    -- Lower volume, heavily queried for charting.
    CREATE TABLE bars (
        symbol      TEXT            NOT NULL,
        interval    TEXT            NOT NULL,  -- '1m', '5m', '1h', '1d'
        open        NUMERIC(18,8)   NOT NULL,
        high        NUMERIC(18,8)   NOT NULL,
        low         NUMERIC(18,8)   NOT NULL,
        close       NUMERIC(18,8)   NOT NULL,
        volume      NUMERIC(18,8)   NOT NULL,
        trade_count INTEGER         NOT NULL,
        vwap        NUMERIC(18,8),
        ts          TIMESTAMPTZ     NOT NULL,
        PRIMARY KEY (symbol, interval, ts)
    );

    -- Primary key IS the index for the main access pattern.
    -- No separate index needed.
    """,

    "timescaledb_hypertable": """
    -- Convert to TimescaleDB hypertable for automatic partitioning.
    -- This replaces manual PARTITION BY RANGE management.
    SELECT create_hypertable('trades', 'ts',
        chunk_time_interval => INTERVAL '1 day',
        if_not_exists => TRUE
    );

    -- Enable compression on old chunks.
    ALTER TABLE trades SET (
        timescaledb.compress,
        timescaledb.compress_segmentby = 'symbol',
        timescaledb.compress_orderby = 'ts DESC'
    );
    SELECT add_compression_policy('trades', INTERVAL '7 days');

    -- Continuous aggregate: auto-maintained 1-minute bars.
    CREATE MATERIALIZED VIEW bars_1m
    WITH (timescaledb.continuous) AS
    SELECT
        symbol,
        time_bucket('1 minute', ts) AS bucket,
        first(price, ts) AS open,
        max(price) AS high,
        min(price) AS low,
        last(price, ts) AS close,
        sum(volume) AS volume,
        count(*) AS trade_count
    FROM trades
    GROUP BY symbol, time_bucket('1 minute', ts)
    WITH NO DATA;

    SELECT add_continuous_aggregate_policy('bars_1m',
        start_offset => INTERVAL '1 hour',
        end_offset   => INTERVAL '1 minute',
        schedule_interval => INTERVAL '1 minute'
    );
    """,
}

for name, sql in timeseries_schema.items():
    print(f"=== {name} ===")
    print(sql)

# Try next:
# 1. Add a continuous aggregate for 1-hour bars.
# 2. Design a retention policy that keeps raw trades for 90 days and 1m bars for 1 year.
# 3. Explain why compress_segmentby = 'symbol' matches the primary query pattern.

## 11. Normalization vs Denormalization

Normalization and denormalization are not opposing philosophies. They are design tools for different access patterns. The right choice depends on whether your workload is write-heavy (OLTP) or read-heavy (analytics/OLAP).

### Normal Forms — What They Mean

**First Normal Form (1NF)** — every column holds atomic (indivisible) values. No arrays, no comma-separated lists, no nested structures in a single column.

Violation: `symbols = 'AAPL,MSFT,GOOG'` in one column.
Fix: one row per symbol, or a proper junction table.

**Second Normal Form (2NF)** — 1NF plus no partial dependencies. Every non-key column depends on the entire primary key, not just part of it. This matters when you have a composite primary key.

Violation: a table with PK `(order_id, product_id)` that also stores `product_name`. The product_name depends only on `product_id`, not on the full key.
Fix: move `product_name` to a separate `products` table.

**Third Normal Form (3NF)** — 2NF plus no transitive dependencies. Every non-key column depends directly on the primary key, not on another non-key column.

Violation: a `trades` table that stores `venue_name` and `venue_country`. The `venue_country` depends on `venue_name`, not on the trade.
Fix: move venue details to a separate `venues` table and reference by `venue_id`.

### When to Normalize

- **OLTP workloads**: frequent inserts, updates, and deletes. Normalization avoids update anomalies (changing a venue name in one place vs hundreds of rows).
- **Data integrity**: single source of truth for each fact.
- **Storage efficiency**: no redundant data.
- **Write-heavy pipelines**: fewer columns to update per row.

### When to Denormalize

- **Analytics / OLAP workloads**: queries join many tables and scan large ranges. Denormalization reduces the number of joins needed.
- **Read-heavy dashboards**: pre-joining data at write time means faster reads.
- **Materialized views and summary tables**: pre-computed results trade storage for speed.

### Decision Framework

| Factor | Normalize | Denormalize |
|---|---|---|
| Write frequency | High | Low |
| Read complexity | Simple lookups | Complex multi-table analytics |
| Data consistency | Critical | Acceptable eventual consistency |
| Update patterns | Entities change frequently | Entities are mostly static |
| Query latency | Can tolerate joins | Needs sub-millisecond response |
| Storage cost | Minimize | Acceptable |

### Practical Reality

Most production systems use both. The OLTP layer (ingestion, order management) is normalized. The analytics layer (dashboards, reports, data warehouse) is denormalized. The ETL/ELT pipeline transforms between the two.

## 12. Star Schema and Dimensional Modeling

Dimensional modeling is the standard approach for analytics data warehouses. The idea: separate measurable events (facts) from descriptive context (dimensions), and structure them for simple, fast joins.

### Fact Tables

A fact table records measurable business events. Each row is one event. The columns are:
- **Foreign keys** to dimension tables (instrument_key, trading_day_key, venue_key)
- **Measures** (price, volume, trade_count, notional_value)

Fact tables are usually the largest tables in the warehouse. They grow continuously.

Fact table types:
- **Transaction fact**: one row per event (one trade, one order)
- **Snapshot fact**: one row per entity per time period (end-of-day position per account)
- **Accumulating snapshot**: one row per process instance, updated as the process progresses (order lifecycle: created, filled, settled)

### Dimension Tables

A dimension table provides descriptive context for the facts. Dimensions are typically small, wide tables with many descriptive attributes.

Examples:
- **dim_instrument**: symbol, name, asset_class, currency, exchange, sector
- **dim_trading_day**: calendar_date, day_of_week, is_holiday, market_session, quarter
- **dim_venue**: venue_code, venue_name, venue_type, country, region

### Star Schema vs Snowflake Schema

**Star schema**: fact table in the center, surrounded by denormalized dimension tables. Each dimension is one JOIN away from the fact table. Simple, fast, easy to understand.

**Snowflake schema**: dimensions are normalized into sub-dimensions. For example, `dim_instrument` links to `dim_asset_class` which links to `dim_sector`. More joins, less redundancy. Harder to query.

**In practice**: star schema is strongly preferred for analytics. The denormalization in dimensions is intentional — it avoids multi-hop joins and makes queries simple. The storage cost of redundancy in dimension tables is trivial compared to the fact table.

### Slowly Changing Dimensions (SCD)

Dimensions change over time. A company changes its name, a symbol gets reassigned, an instrument moves to a different asset class. How you handle these changes matters for historical accuracy.

**SCD Type 1 — Overwrite.** Replace the old value with the new value. Simple but destroys history. Use when history does not matter (fixing a typo).

```sql
-- SCD Type 1: just UPDATE
UPDATE dim_instrument SET symbol = 'META' WHERE symbol = 'FB';
-- Historical trades now show 'META' even though they happened when it was 'FB'.
```

**SCD Type 2 — Add a new row with effective dates.** The old row gets an `effective_end` date, and a new row is created with an `effective_start` date. This preserves full history.

```sql
-- SCD Type 2: the instrument dimension has effective dates.
-- When querying historical trades, JOIN on the dimension that was
-- active at the time of the trade.
```

**Why SCD Type 2 matters**: in trading, instruments change. Symbols get reassigned (AAPL was Apple Records before it was Apple Inc). Companies rebrand. Currencies redenominate. If you overwrite the dimension, historical reports become wrong. SCD Type 2 ensures that a trade from 2020 is attributed to the entity that existed in 2020, not whatever the entity looks like today.

In [ ]:
# Star Schema for a Trading Analytics Data Warehouse
#
# This is the complete dimensional model. The fact table (fact_trades)
# is surrounded by dimension tables, each one JOIN away.

star_schema = {
    "fact_trades": {
        "columns": [
            "trade_key        BIGSERIAL PRIMARY KEY",
            "instrument_key   INTEGER NOT NULL REFERENCES dim_instrument(instrument_key)",
            "trading_day_key  INTEGER NOT NULL REFERENCES dim_trading_day(trading_day_key)",
            "venue_key        INTEGER NOT NULL REFERENCES dim_venue(venue_key)",
            "trade_id         UUID NOT NULL",
            "price            NUMERIC(18,8) NOT NULL",
            "volume           NUMERIC(18,8) NOT NULL",
            "notional_value   NUMERIC(18,8) NOT NULL",  # price * volume, pre-computed
            "side             TEXT NOT NULL",
            "ts               TIMESTAMPTZ NOT NULL",
        ],
        "indexes": [
            "CREATE INDEX idx_ft_instrument_day ON fact_trades (instrument_key, trading_day_key);",
            "CREATE INDEX idx_ft_ts ON fact_trades (ts);",
        ],
        "notes": (
            "One row per trade. Grain = individual trade event. "
            "Foreign keys link to dimensions for descriptive context. "
            "notional_value is pre-computed to avoid repeated multiplication in queries."
        ),
    },

    "dim_instrument_scd2": {
        "columns": [
            "instrument_key   SERIAL PRIMARY KEY",
            "instrument_id    TEXT NOT NULL",        # business key (stable across versions)
            "symbol           TEXT NOT NULL",         # can change over time
            "name             TEXT",
            "asset_class      TEXT NOT NULL",         # e.g., 'equity', 'commodity', 'fx'
            "currency         TEXT NOT NULL",
            "exchange         TEXT",
            "sector           TEXT",
            "effective_start  DATE NOT NULL",
            "effective_end    DATE NOT NULL DEFAULT '9999-12-31'",
            "is_current       BOOLEAN NOT NULL DEFAULT TRUE",
        ],
        "indexes": [
            "CREATE INDEX idx_di_instrument_id ON dim_instrument (instrument_id);",
            "CREATE INDEX idx_di_current ON dim_instrument (instrument_id) WHERE is_current;",
        ],
        "notes": (
            "SCD Type 2: when an instrument's attributes change, the current row gets "
            "effective_end = today and is_current = FALSE, and a new row is inserted with "
            "effective_start = today and is_current = TRUE. "
            "instrument_key is a surrogate key (auto-increment). "
            "instrument_id is the business key (stable across versions). "
            "To join historical trades to the correct instrument version: "
            "JOIN dim_instrument d ON f.instrument_key = d.instrument_key "
            "-- The ETL assigns the correct instrument_key at load time based on effective dates."
        ),
    },

    "dim_trading_day": {
        "columns": [
            "trading_day_key  SERIAL PRIMARY KEY",
            "calendar_date    DATE NOT NULL UNIQUE",
            "day_of_week      TEXT NOT NULL",         # 'Monday', 'Tuesday', ...
            "is_weekend       BOOLEAN NOT NULL",
            "is_holiday       BOOLEAN NOT NULL",
            "market_session   TEXT",                   # 'REGULAR', 'EXTENDED', NULL if holiday
            "quarter          TEXT NOT NULL",           # 'Q1', 'Q2', 'Q3', 'Q4'
            "fiscal_year      INTEGER NOT NULL",
        ],
        "notes": (
            "One row per calendar date. Pre-populated for the full date range. "
            "Avoids date math in queries — just filter on is_holiday or quarter."
        ),
    },

    "dim_venue": {
        "columns": [
            "venue_key        SERIAL PRIMARY KEY",
            "venue_code       TEXT NOT NULL UNIQUE",
            "venue_name       TEXT NOT NULL",
            "venue_type       TEXT NOT NULL",          # 'ECN', 'OTC', 'EXCHANGE', 'DARK_POOL'
            "country          TEXT NOT NULL",
            "region           TEXT NOT NULL",           # 'NA', 'EMEA', 'APAC'
        ],
        "notes": (
            "Small, mostly static dimension. SCD Type 1 (overwrite) is fine here "
            "because venue attributes rarely change in ways that affect historical analysis."
        ),
    },
}

# Print the schema
for table_name, table_def in star_schema.items():
    print(f"=== {table_name} ===")
    if "columns" in table_def:
        print("  Columns:")
        for col in table_def["columns"]:
            print(f"    {col}")
    if "indexes" in table_def:
        print("  Indexes:")
        for idx in table_def["indexes"]:
            print(f"    {idx}")
    print(f"  Notes: {table_def['notes']}")
    print()

# Example query: total notional by asset class and quarter.
example_query = """
-- Star schema query: total notional value by asset class and quarter.
-- Notice: only simple one-hop JOINs from fact to dimensions.
SELECT
    d.asset_class,
    td.quarter,
    td.fiscal_year,
    SUM(f.notional_value) AS total_notional,
    COUNT(*) AS trade_count,
    AVG(f.price) AS avg_price
FROM fact_trades f
JOIN dim_instrument d ON f.instrument_key = d.instrument_key
JOIN dim_trading_day td ON f.trading_day_key = td.trading_day_key
WHERE td.fiscal_year = 2026
  AND d.is_current = TRUE
GROUP BY d.asset_class, td.quarter, td.fiscal_year
ORDER BY td.quarter, total_notional DESC;
"""

# SCD Type 2 query: find the instrument version active at trade time.
scd2_query = """
-- SCD Type 2: join trades to the instrument version that was active
-- at the time of the trade. The ETL pre-assigns instrument_key,
-- but if you need to look it up at query time:
SELECT
    f.trade_id,
    f.price,
    f.ts,
    d.symbol,
    d.name,
    d.asset_class
FROM fact_trades f
JOIN dim_instrument d ON f.instrument_key = d.instrument_key;
-- Because instrument_key is a surrogate pointing to a specific
-- SCD2 version, this join always returns the historically correct version.

-- If you only have instrument_id and ts (no pre-assigned key):
SELECT
    t.trade_id,
    t.price,
    t.ts,
    d.symbol,
    d.name
FROM trades t
JOIN dim_instrument d
    ON t.instrument_id = d.instrument_id
    AND t.ts >= d.effective_start
    AND t.ts < d.effective_end;
"""

print("=== Example Star Schema Query ===")
print(example_query)
print("=== SCD Type 2 Join ===")
print(scd2_query)

# Try next:
# 1. Add a dim_account dimension for portfolio tracking.
# 2. Write a query that compares this quarter to last quarter by asset class.
# 3. Explain why the ETL should pre-assign instrument_key instead of joining on effective dates at query time.

## 13. Mini Lab

Work through these exercises against a real PostgreSQL instance. They combine the concepts from this notebook into realistic tasks.

### Exercise 1: Design a Complete Trading Data Warehouse Schema

Design the following:
- A `trades` table partitioned by month with proper indexes
- A `dim_instrument` table with SCD Type 2
- A `dim_trading_day` table pre-populated for 2026
- A `fact_daily_summary` snapshot fact table (one row per instrument per day)

Justify every column, index, and design choice.

### Exercise 2: Write 5 Queries Using Window Functions, CTEs, and Reconciliation Patterns

1. **Top-3 trades by volume per symbol today** — use ROW_NUMBER in a CTE
2. **Gap detection** — find symbols with no trades for more than 10 minutes using LAG in a CTE
3. **VWAP per symbol per hour** — use window functions with time bucketing
4. **Vendor reconciliation** — find trades in vendor A but not vendor B using LEFT JOIN WHERE NULL
5. **Price anomaly detection** — find trades where price deviates more than 3 standard deviations from the rolling 20-trade average, using window frames and CTEs

### Exercise 3: Read an EXPLAIN Output and Propose an Optimization

Given this plan:

```
Sort  (cost=89234.56..89245.67 rows=50000 width=64) (actual time=1523.4..1534.2 rows=48750 loops=1)
  Sort Key: ts DESC
  Sort Method: external merge  Disk: 4096kB
  ->  Hash Join  (cost=1234.56..78901.23 rows=50000 width=64) (actual time=45.2..1389.5 rows=48750 loops=1)
        Hash Cond: (t.instrument_id = d.instrument_id)
        ->  Seq Scan on trades t  (cost=0.00..67890.00 rows=50000 width=52) (actual time=0.03..1201.3 rows=48750 loops=1)
              Filter: ((ts >= '2026-01-01') AND (ts < '2026-02-01'))
              Rows Removed by Filter: 2951250
        ->  Hash  (cost=1000.00..1000.00 rows=500 width=16) (actual time=1.2..1.2 rows=500 loops=1)
              ->  Seq Scan on dim_instrument d  (cost=0.00..1000.00 rows=500 width=16) (actual time=0.01..0.8 rows=500 loops=1)
Planning Time: 0.5 ms
Execution Time: 1540.1 ms
```

Questions to answer:
1. What is the dominant bottleneck?
2. What index would you create?
3. What would the improved plan look like?
4. Would the Sort spill to disk be fixed by the index?

In [ ]:
# Mini Lab Exercise 3 — Answer Key
#
# Walk through the EXPLAIN output step by step.

explain_analysis = {
    "1_dominant_bottleneck": (
        "Seq Scan on trades: reading 3M rows (48750 + 2951250 removed by filter) "
        "to find 48750. The filter removes 98.4% of rows scanned. "
        "This is a full table scan where an index scan would be far more efficient."
    ),
    "2_recommended_index": (
        "CREATE INDEX idx_trades_ts ON trades (ts);\n"
        "Or better, if partitioned by time: partition pruning would eliminate "
        "the need to scan non-January partitions entirely.\n"
        "If queries also filter by instrument_id: "
        "CREATE INDEX idx_trades_ts_instrument ON trades (ts, instrument_id);"
    ),
    "3_improved_plan": (
        "With the index, expect:\n"
        "  Index Scan Backward using idx_trades_ts on trades\n"
        "    Index Cond: ((ts >= '2026-01-01') AND (ts < '2026-02-01'))\n"
        "  -> Nested Loop or Hash Join with dim_instrument\n"
        "The Seq Scan disappears. Only 48750 rows are read from the index.\n"
        "If using partitioning: only the January partition is scanned."
    ),
    "4_sort_spill_fix": (
        "The Sort spills to disk because work_mem is too small for 48750 rows. "
        "Two fixes:\n"
        "  a) Increase work_mem: SET work_mem = '16MB'; (session-level)\n"
        "  b) If the index matches ORDER BY ts DESC, the Index Scan Backward "
        "     eliminates the Sort node entirely — no sort needed at all.\n"
        "Fix (b) is strictly better because it removes the sort rather than "
        "just making it fit in memory."
    ),
}

for question, answer in explain_analysis.items():
    print(f"\n{question}:")
    print(f"  {answer}")

# Mini Lab Exercise 2 — Query Templates
mini_lab_queries = {
    "1_top_3_by_volume": """
    WITH ranked AS (
        SELECT
            trade_id, symbol, price, volume, ts,
            ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY volume DESC) AS rn
        FROM trades
        WHERE ts >= CURRENT_DATE
    )
    SELECT trade_id, symbol, price, volume, ts
    FROM ranked
    WHERE rn <= 3
    ORDER BY symbol, volume DESC;
    """,

    "2_gap_detection": """
    WITH trade_times AS (
        SELECT
            symbol,
            ts,
            LAG(ts) OVER (PARTITION BY symbol ORDER BY ts) AS prev_ts
        FROM trades
        WHERE ts >= CURRENT_DATE
    )
    SELECT symbol, ts, prev_ts, ts - prev_ts AS gap
    FROM trade_times
    WHERE ts - prev_ts > INTERVAL '10 minutes'
    ORDER BY gap DESC;
    """,

    "3_vwap_per_hour": """
    SELECT
        symbol,
        date_trunc('hour', ts) AS hour_bucket,
        SUM(price * volume) / NULLIF(SUM(volume), 0) AS vwap,
        SUM(volume) AS total_volume,
        COUNT(*) AS trade_count
    FROM trades
    WHERE ts >= CURRENT_DATE
    GROUP BY symbol, date_trunc('hour', ts)
    ORDER BY symbol, hour_bucket;
    """,

    "4_vendor_reconciliation": """
    SELECT a.trade_id, a.symbol, a.price, a.ts
    FROM vendor_a_trades a
    LEFT JOIN vendor_b_trades b ON a.trade_id = b.trade_id
    WHERE b.trade_id IS NULL
    ORDER BY a.ts;
    """,

    "5_price_anomaly_detection": """
    WITH rolling_stats AS (
        SELECT
            trade_id, symbol, price, ts,
            AVG(price) OVER (
                PARTITION BY symbol ORDER BY ts
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS rolling_avg,
            STDDEV(price) OVER (
                PARTITION BY symbol ORDER BY ts
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS rolling_stddev,
            COUNT(*) OVER (
                PARTITION BY symbol ORDER BY ts
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS window_size
        FROM trades
        WHERE ts >= CURRENT_DATE
    )
    SELECT trade_id, symbol, price, ts, rolling_avg, rolling_stddev,
           (price - rolling_avg) / NULLIF(rolling_stddev, 0) AS z_score
    FROM rolling_stats
    WHERE window_size >= 20
      AND ABS((price - rolling_avg) / NULLIF(rolling_stddev, 0)) > 3
    ORDER BY ABS((price - rolling_avg) / NULLIF(rolling_stddev, 0)) DESC;
    """,
}

print("\n\n=== MINI LAB QUERY TEMPLATES ===")
for name, sql in mini_lab_queries.items():
    print(f"\n--- {name} ---")
    print(sql)

## 14. Exit Criteria

Do not move on until you can say yes to these without searching:

- I can explain the SQL logical execution order (FROM through LIMIT) and why you cannot use a SELECT alias in WHERE.
- I can write all join types correctly and explain when to use each.
- I can use the LEFT JOIN WHERE NULL pattern for data reconciliation.
- I can write window functions using ROW_NUMBER, RANK, LAG, and window frames.
- I can explain PARTITION BY vs GROUP BY and when to use which.
- I can use CTEs for readable multi-step queries and for filtering on window functions.
- I can read an EXPLAIN ANALYZE output, identify scan types and join strategies, and spot the dominant bottleneck.
- I can design composite indexes with correct column ordering (equality first, range last).
- I can explain when indexes help and when they hurt.
- I can explain MVCC: readers do not block writers, snapshot isolation, dead tuples, VACUUM.
- I can explain WAL: crash recovery, checkpoints, and the connection to replication.
- I can design a partitioned table and explain when partitioning helps and when it hurts.
- I can design a time-series schema with defensible index and partitioning choices.
- I can explain 1NF, 2NF, 3NF and when to normalize vs denormalize.
- I can design a star schema with fact tables, dimension tables, and SCD Type 2.
- I can explain why SCD Type 2 matters for historical correctness in trading data.

## 15. References

- PostgreSQL Documentation: SQL Syntax — SELECT
  https://www.postgresql.org/docs/current/sql-select.html
- PostgreSQL Documentation: Query Planning — Using EXPLAIN
  https://www.postgresql.org/docs/current/using-explain.html
- PostgreSQL Documentation: Indexes
  https://www.postgresql.org/docs/current/indexes.html
- PostgreSQL Documentation: Concurrency Control (MVCC)
  https://www.postgresql.org/docs/current/mvcc.html
- PostgreSQL Documentation: Write-Ahead Logging (WAL)
  https://www.postgresql.org/docs/current/wal.html
- PostgreSQL Documentation: Table Partitioning
  https://www.postgresql.org/docs/current/ddl-partitioning.html
- PostgreSQL Documentation: Window Functions
  https://www.postgresql.org/docs/current/tutorial-window.html
- PostgreSQL Documentation: Window Function Processing
  https://www.postgresql.org/docs/current/sql-expressions.html#SYNTAX-WINDOW-FUNCTIONS
- PostgreSQL Documentation: Routine Vacuuming
  https://www.postgresql.org/docs/current/routine-vacuuming.html
- TimescaleDB Documentation: Hypertables
  https://docs.timescale.com/use-timescale/latest/hypertables/
- TimescaleDB Documentation: Compression
  https://docs.timescale.com/use-timescale/latest/compression/
- TimescaleDB Documentation: Continuous Aggregates
  https://docs.timescale.com/use-timescale/latest/continuous-aggregates/
- Ralph Kimball, *The Data Warehouse Toolkit* — dimensional modeling, star schema, SCD types

## Interview Question Bank

Use these after you finish the notebook. Answer them from memory.

- What is the logical execution order of a SQL query, and why does that matter?
- Why can you not use a `SELECT` alias in `WHERE`?
- When would you use `HAVING` instead of `WHERE`?
- What is the difference between `GROUP BY` and `PARTITION BY`?
- How do you reason about whether an index will actually help a query?
- What does MVCC buy you in PostgreSQL?
- What is WAL, and why does it matter for crash recovery and replication?
- Why would you choose SCD Type 2 for a trading-data dimension table?
